# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/satkynovilgiz/flyrank-ml-content-refresh/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Binary classification, used as a ranking / scoring problem.**

- The model learns a yes/no label: is this page declining (1) or not (0)?
- I don't use the yes/no answer itself. I use the model's **probability** as a score.
- I sort pages by that score, high to low. That sorted list is the editor's review queue.

Why ranking and not plain classification? My question is "which pages **first**?"
An editor never sees all 30,000 pages. They work from the top of a list.
So what matters is the **order** of the list, not a 0/1 call on every page.


In [1]:
# Setup: find the repo root (works from work/notebooks/, the repo root, or Colab) and load the starter CSV.
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and not Path("flyrank-ml-internship-starter").is_dir():
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/flyrank-bih/flyrank-ml-internship-starter"], check=True)
    os.chdir("flyrank-ml-internship-starter")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
root = Path.cwd()
while not (root / "data/raw/content_refresh_anonymized.csv").exists() and root != root.parent:
    root = root.parent
os.chdir(root)

import json
import numpy as np, pandas as pd
from sklearn.metrics import average_precision_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.tree import DecisionTreeClassifier

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
y = df["is_declining_label"].to_numpy()

# Task type check: a binary label, but the output we use is a score that ORDERS pages.
print("label values:", sorted(df["is_declining_label"].unique().tolist()), "-> binary classification")
print("the model's predict_proba score is then sorted high -> low to build the review queue")


label values: [0, 1] -> binary classification
the model's predict_proba score is then sorted high -> low to build the review queue


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Target: `is_declining_label` (1 when `trend_direction == "down"`).**
I use it as a **proxy** for what I really care about, which is "this page is worth an editor's review."

Where it comes from (checked in the cell below):

- The label is **defined by a rule**, not observed as a later outcome.
- `trend_direction` is "down" when impressions in the last 30 days fell more than 20% versus the 30 days before.
  The table shows "down" pages have `trend_pct` from -100 to -20, and every one of them has label 1.
- Recomputing the label from `impressions_last_30d` vs `impressions_prev_30d` matches it on 100.00% of pages.
  So those columns, plus `trend_direction` and `trend_pct`, can never be features.

Why it is **only a proxy** (guide section 7):

- Declining does not mean a refresh will pay off.
- A drop can be **seasonality** (demand fell for everyone), **consolidation** (a sibling page took the traffic), a **SERP / AI click change**, or plain **noise** on a low-volume page.
- It is measured on the **current** window, not a future one. A stronger target would be "features from an earlier window → decline over the next 30 days."
- One more caveat I see: `impressions_90d` is a feature, and its 90 days include the label's last 30 days. The windows overlap. I'll check this properly in my ML-04 data contract.

For now the proxy is good enough to frame the task and beat a baseline honestly. It is not the final word.


In [2]:
# Where the label comes from: it is DEFINED from trend_direction, which is defined from trend_pct,
# which is computed from impressions_last_30d vs impressions_prev_30d. Aggregates only.
label_src = (df.groupby(df["trend_direction"].str.lower())
               .agg(pages=("is_declining_label", "size"),
                    label_mean=("is_declining_label", "mean"),
                    trend_pct_min=("trend_pct", "min"),
                    trend_pct_max=("trend_pct", "max"))
               .sort_values("pages", ascending=False))
print(label_src.round(1).to_string())

# Recompute the label from the two 30-day impression columns, with the dictionary's rule (down < -20%).
prev, last = df["impressions_prev_30d"], df["impressions_last_30d"]
recomputed = ((prev > 0) & ((last - prev) / prev.where(prev > 0) * 100 < -20)).astype(int)
print(f"\nlabel recomputed from impressions_last_30d vs impressions_prev_30d "
      f"matches is_declining_label on {(recomputed == df['is_declining_label']).mean():.2%} of pages")


                 pages  label_mean  trend_pct_min  trend_pct_max
trend_direction                                                 
down             16262         1.0         -100.0          -20.0
stable            5962         0.0          -20.0           20.0
up                4388         0.0           20.0        44900.0
new               2236         0.0            NaN            NaN
flat              1152         0.0            NaN            NaN

label recomputed from impressions_last_30d vs impressions_prev_30d matches is_declining_label on 100.00% of pages


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Main metric: Precision@50 on held-out clients.** Supporting metric: average precision.

- **Precision@50** = of the top 50 pages in my queue, the share that are actually declining.
- **Held-out clients** = I split with `GroupShuffleSplit` by `client_id` (20% of clients, seed 42). The model is tested on clients it never saw. That is the real situation for a new FlyRank client.
- **Average precision** checks the whole ranking, not just the top 50. That way I can't win by luck in one small window.

Why precision at the top and not accuracy:
editors only review the top K pages.
Accuracy also counts every page at the bottom of the list that nobody will ever open.
A model can have fine accuracy and still put the wrong pages at the top.
Precision@50 measures exactly the pages that cost editor time.

**What "good" means** (numbers from the cell below):

- **Base rate on the held-out clients: 0.511.** A random queue already gets about half right. Anything near 0.511 is no better than random.
- **Hand rule on the same held-out clients:** P@50 0.440, and 0.511 tie-robust. It flags no pages there, so its "ranking" is just tie order.
- **Pipeline baseline (`scripts/run_all.py`, its own client holdout of 2,325 test pages): P@50 0.240, average precision 0.468.** This is the baseline number I must beat.
- On that same pipeline split, the random forest reached P@50 0.680 (average precision 0.610).

So "good" for me means clearly above **both** the base rate and the hand-rule baseline on held-out clients, and holding up across repeated client splits (that check comes in ML-09).


In [3]:
# Success metric: Precision@50 on HELD-OUT CLIENTS, plus average precision, vs the base rate and the hand rule.
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

def expected_precision_at_k(scores, labels, k):
    """Precision@K if tied scores were broken at random (tie-robust)."""
    scores, labels = np.asarray(scores), np.asarray(labels)
    hits, left = 0.0, k
    for s in np.sort(np.unique(scores))[::-1]:
        block = labels[scores == s]
        take = min(left, len(block))
        hits += take * block.mean()
        left -= take
        if left == 0:
            break
    return hits / k

# Client holdout: GroupShuffleSplit by client_id, 20% of clients held out, seed 42 (same as ML-01).
groups = df["client_id"].to_numpy()
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(df, y, groups=groups))
assert not set(groups[train_idx]) & set(groups[test_idx]), "a client is in both train and test"
y_te = y[test_idx]

# The ML-01 hand rule: stale (>= 180 d since update) AND visible (>= 500 impressions), scored by impressions.
hand_score = (((df["days_since_last_update"] >= 180) & (df["impressions_90d"] >= 500))
              * df["impressions_90d"]).to_numpy()

print(f"held-out: {len(set(groups[test_idx]))} clients, {len(test_idx):,} pages")
print(f"base rate (random queue) on held-out clients: {y_te.mean():.3f}")
print(f"hand rule on held-out clients: P@50 {precision_at_k(hand_score[test_idx], y_te, 50):.3f} | "
      f"P@50 tie-robust {expected_precision_at_k(hand_score[test_idx], y_te, 50):.3f} | "
      f"avg precision {average_precision_score(y_te, hand_score[test_idx]):.3f}")

# The pipeline's own client-holdout receipt (written by `python scripts/run_all.py`; gitignored).
results_path = Path("outputs/model_results.json")
if results_path.exists():
    res = json.loads(results_path.read_text())
    print(f"\npipeline ({res['split_strategy']}, {res['test_rows']:,} test pages):")
    print(f"  baseline_rules      P@50 {res['baseline']['baseline_precision_at_50']:.3f} | "
          f"avg precision {res['baseline']['baseline_average_precision']:.3f}")
    for name, m in res["models"].items():
        print(f"  {name:<19} P@50 {m['precision_at_50']:.3f} | avg precision {m['average_precision']:.3f}")
else:
    print("\noutputs/model_results.json not found: run `python scripts/run_all.py` to add the pipeline receipt")


held-out: 7 clients, 6,163 pages
base rate (random queue) on held-out clients: 0.511
hand rule on held-out clients: P@50 0.440 | P@50 tie-robust 0.511 | avg precision 0.511

pipeline (client_holdout, 2,325 test pages):
  baseline_rules      P@50 0.240 | avg precision 0.468
  decision_tree       P@50 0.540 | avg precision 0.575
  logistic_regression P@50 0.400 | avg precision 0.522
  random_forest       P@50 0.680 | avg precision 0.610


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**One row = one content page (`content_id`). Pages are grouped by `client_id`.**

From the cell below:

- **30,000 rows**, and 30,000 unique `content_id` values. The grain check passes: no page appears twice.
- **32 clients.** Pages per client: **min 3, median 567, max 7,008.** Very uneven. A few big clients could dominate a random split, which is one more reason to split by client.
- **Target:** 16,262 pages are 1 (**54.2%**) and 13,738 are 0 (45.8%). Fairly balanced.
- The preview shows a few non-ID columns and the target only. I never print `content_id` or `client_id`.

Column roles (the cell prints the full lists):

- **Forbidden, they define the label:** `trend_direction`, `trend_pct`, `impressions_last_30d`, `impressions_prev_30d`.
- **Forbidden, same 30-day windows as the label:** the `clicks_` and `sessions_` last/prev 30-day columns.
- **Grouping only:** `content_id`, `client_id`. They are pseudonyms, used only for the split.
- **Features:** the pipeline's 18 numeric and 8 categorical features, read straight from `scripts/ml_utils.py`. The cell asserts that no forbidden column is in that list.


In [4]:
# Unit of analysis: one row = one content page (content_id); pages are grouped by client_id.
print(f"rows: {len(df):,} | unique content_id: {df['content_id'].nunique():,} | clients: {df['client_id'].nunique()}")
assert df["content_id"].is_unique, "grain check failed: a content_id appears twice"

ppc = df.groupby("client_id").size()
print(f"pages per client: min {ppc.min():,} | median {ppc.median():,.0f} | max {ppc.max():,}")

# Safe preview: a few non-ID columns plus the target. No content_id / client_id printed.
preview_cols = ["content_type", "content_age_days", "days_since_last_update",
                "impressions_90d", "avg_position", "ctr", "is_declining_label"]
display(df[preview_cols].sample(5, random_state=0).reset_index(drop=True))

# What the target looks like.
counts = df["is_declining_label"].value_counts().sort_index()
print(pd.DataFrame({"pages": counts, "share": (counts / len(df)).round(3)}).to_string())

# Features vs forbidden columns. Feature lists are read from the pipeline (scripts/ml_utils.py), not copied.
sys.path.insert(0, str(Path("scripts").resolve()))
from ml_utils import MODEL_NUMERIC_FEATURES, MODEL_CATEGORICAL_FEATURES

roles = {
    "FORBIDDEN: defines the label": ["trend_direction", "trend_pct",
                                      "impressions_last_30d", "impressions_prev_30d"],
    "FORBIDDEN: same 30-day windows as the label": ["clicks_last_30d", "clicks_prev_30d",
                                                     "sessions_last_30d", "sessions_prev_30d"],
    "GROUPING ONLY: pseudonymous IDs": ["content_id", "client_id"],
    "TARGET": ["is_declining_label"],
    "FEATURE (numeric, pipeline)": MODEL_NUMERIC_FEATURES,
    "FEATURE (categorical, pipeline)": MODEL_CATEGORICAL_FEATURES,
}
pipeline_inputs = {"impressions_90d": "log_impressions_90d", "clicks_90d": "log_clicks_90d",
                   "sessions_90d": "log_sessions_90d", "ai_sessions_90d": "log_ai_sessions_90d"}
forbidden = set(sum(list(roles.values())[:4], []))
features = set(MODEL_NUMERIC_FEATURES) | set(MODEL_CATEGORICAL_FEATURES)
assert not forbidden & features, "a forbidden column is in the feature list"
assigned = forbidden | features | set(pipeline_inputs)
roles["FEATURE input (logged by the pipeline)"] = list(pipeline_inputs)
roles["NOT USED YET (raw, not in the pipeline)"] = [c for c in df.columns if c not in assigned]
for role, cols in roles.items():
    print(f"\n{role} ({len(cols)}):\n  " + ", ".join(cols))


rows: 30,000 | unique content_id: 30,000 | clients: 32
pages per client: min 3 | median 567 | max 7,008


,content_type,content_age_days,days_since_last_update,impressions_90d,avg_position,ctr,is_declining_label
0,keyword article,148,104,5718,18.4,0.16,1
1,keyword article,421,7,525,19.7,0.00,1
2,keyword article,139,15,70993,6.0,0.31,0
3,keyword article,98,20,3157,12.6,0.29,1
4,keyword article,463,22,604,29.3,0.17,0


                    pages  share
is_declining_label              
0                   13738  0.458
1                   16262  0.542



FORBIDDEN: defines the label (4):
  trend_direction, trend_pct, impressions_last_30d, impressions_prev_30d

FORBIDDEN: same 30-day windows as the label (4):
  clicks_last_30d, clicks_prev_30d, sessions_last_30d, sessions_prev_30d

GROUPING ONLY: pseudonymous IDs (2):
  content_id, client_id

TARGET (1):
  is_declining_label

FEATURE (numeric, pipeline) (18):
  search_volume, competition, cpc, word_count, char_count, log_impressions_90d, log_clicks_90d, log_sessions_90d, log_ai_sessions_90d, days_with_impressions, days_with_sessions, content_age_days, days_since_last_update, ctr, avg_position, engagement_rate, scroll_rate, ai_traffic_pct

FEATURE (categorical, pipeline) (8):
  competition_level, content_type, main_intent, age_tier, freshness_tier, word_count_tier, impression_tier, position_tier

FEATURE input (logged by the pipeline) (4):
  impressions_90d, clicks_90d, sessions_90d, ai_sessions_90d

NOT USED YET (raw, not in the pipeline) (8):
  provider_used, model_used, pageviews_90d

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

The hand rule ("stale for 180+ days AND 500+ impressions") barely fires in this data.
`days_since_last_update >= 180` is true for only 0.58% of pages.
So the rule flags 17 of 30,000 pages (0.06%), from just 4 of 32 clients, and 0 pages on the held-out clients.
For almost every client, the rule gives every page the same score, so the queue order is random.

The signals also interact, and one threshold can't capture that.
The share trending down by age depends on visibility.
For moderate-impression pages it falls from 0.713 (90-179 days) to 0.440 (365+ days).
For excellent-impression pages it barely moves (0.464 / 0.483 / 0.421).
Position and CTR change the picture again inside those groups.
A rule would need a different threshold for every combination. A tree or forest learns those splits from data.

On the same held-out clients (base rate 0.511), a depth-3 tree on six pre-decision features reached P@50 0.60 (0.589 tie-robust, average precision 0.567).
The hand rule reached 0.44 (0.511 tie-robust, average precision 0.511), no better than random.
In the pipeline's client holdout, the random forest reached P@50 0.680 vs 0.240 for the baseline rules.
These are single splits on a 30,000-row slice, so I treat them as directional until ML-09 repeats them.

The output is tied to a real content action.
It is a **ranked review queue** where each page gets a **suggested action** (refresh, expand, protect, prune, monitor) and a **reason code** an editor can check, for example `declining_with_demand` or `page_one_decay_risk`.
The model decides the order. The editor still decides the edit.


In [5]:
# Why a fixed rule is not enough. All numbers on aggregates.
# 1) Coverage: how many pages and clients can the hand rule rank at all?
flag = hand_score > 0
flag_clients = df.loc[flag, "client_id"].nunique()
print(f"hand rule flags {flag.sum()} of {len(df):,} pages ({flag.mean():.2%}), "
      f"from {flag_clients} of {df['client_id'].nunique()} clients; held-out clients: {flag[test_idx].sum()} pages")
print(f"days_since_last_update >= 180 is true for {(df['days_since_last_update'] >= 180).mean():.2%} of pages\n")

# 2) Signals interact: the share trending down by age band x impression tier.
age_band = pd.cut(df["content_age_days"], bins=[90, 180, 365, np.inf],
                  labels=["90-179 d", "180-364 d", "365+ d"], right=False)
tiers = ["low", "moderate", "good", "excellent"]
inter = (df.assign(age_band=age_band)
           .pivot_table(index="age_band", columns="impression_tier", values="is_declining_label",
                        aggfunc="mean", observed=False)[tiers])
n_cell = df.assign(age_band=age_band).pivot_table(index="age_band", columns="impression_tier",
                     values="is_declining_label", aggfunc="size", observed=False)[tiers]
print("share trending down by age band (rows) x impression tier (columns):")
print(inter.round(3).to_string())
print(f"smallest cell: {int(n_cell.min().min()):,} pages\n")

# 3) A learned ranker on the same held-out clients (the ML-01 depth-3 tree, six pre-decision features).
tree_feats = ["content_age_days", "days_since_last_update", "impressions_90d",
              "avg_position", "ctr", "word_count"]
assert not forbidden & set(tree_feats)
X = df[tree_feats].replace([np.inf, -np.inf], np.nan).fillna(0)
tree = DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=42).fit(
           X.iloc[train_idx], y[train_idx])
s_te = tree.predict_proba(X.iloc[test_idx])[:, 1]
summary = pd.DataFrame({
    "P@50":            [precision_at_k(hand_score[test_idx], y_te, 50), precision_at_k(s_te, y_te, 50)],
    "P@50 tie-robust": [expected_precision_at_k(hand_score[test_idx], y_te, 50), expected_precision_at_k(s_te, y_te, 50)],
    "avg precision":   [average_precision_score(y_te, hand_score[test_idx]), average_precision_score(y_te, s_te)],
}, index=["hand rule", "tree depth 3"])
print(f"held-out clients, base rate {y_te.mean():.3f}:")
print(summary.round(3).to_string())


hand rule flags 17 of 30,000 pages (0.06%), from 4 of 32 clients; held-out clients: 0 pages
days_since_last_update >= 180 is true for 0.58% of pages



share trending down by age band (rows) x impression tier (columns):
impression_tier    low  moderate   good  excellent
age_band                                          
90-179 d         0.530     0.713  0.667      0.464
180-364 d        0.406     0.640  0.576      0.483
365+ d           0.401     0.440  0.434      0.421
smallest cell: 240 pages



held-out clients, base rate 0.511:
              P@50  P@50 tie-robust  avg precision
hand rule     0.44            0.511          0.511
tree depth 3  0.60            0.589          0.567


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.